# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [7]:
class Matrix:
    # It can initialize with Matrix(n, m) or Matrix([[...], [...]])
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            rows, cols = args
            self._rows = rows
            self._cols = cols
            self._data = [[0 for _ in range(cols)] for _ in range(rows)]

        elif len(args) == 1 and isinstance(args[0], list):
            values = args[0]

            if len(values) == 0:
                self._data = []
                self._rows = 0
                self._cols = 0
                return

            col_len = len(values[0])

            # It make sure each row has the same number of entries
            for row in values:
                if not isinstance(row, list) or len(row) != col_len:
                    raise ValueError("Matrix improperly specified")

            self._data = [row[:] for row in values]
            self._rows = len(values)
            self._cols = col_len

        else:
            raise TypeError("Invalid arguments for Matrix initialization")

    # It supports both M[i][j] and M[i, j]
    def __getitem__(self, index):
        if isinstance(index, tuple):
            i, j = index
            return self._data[i][j]
        return self._data[index]

    def set_values(self, other):
        if isinstance(other, Matrix):
            new_data = other._data
            new_rows = other._rows
            new_cols = other._cols

        elif isinstance(other, list):
            new_rows = len(other)
            new_cols = len(other[0]) if new_rows > 0 else 0
            new_data = other

        else:
            raise TypeError("Can only assign from Matrix or list of lists")

        if new_rows != self._rows or new_cols != self._cols:
            raise ValueError("Dimensions do not match for assignment")

        self._data = [row[:] for row in new_data]

    def __str__(self):
        return "\n".join(str(row) for row in self._data)


m1 = Matrix(2, 3)
print("--- Matrix(2, 3) ---")
print(m1)

m2 = Matrix([[1, 2], [3, 4]])
print("\n--- Matrix from list ---")
print(m2)

print("\n--- Checking indexing ---")
print("m2[0,1] =", m2[0, 1])
print("m2[0][1] =", m2[0][1])

m2.set_values([[10, 20], [30, 40]])
print("\n--- After set_values ---")
print(m2)

try:
    bad = Matrix([[1, 2], [3]])
except ValueError as e:
    print("\n--- Improper matrix test ---")
    print(e)

--- Matrix(2, 3) ---
[0, 0, 0]
[0, 0, 0]

--- Matrix from list ---
[1, 2]
[3, 4]

--- Checking indexing ---
m2[0,1] = 2
m2[0][1] = 2

--- After set_values ---
[10, 20]
[30, 40]

--- Improper matrix test ---
Matrix improperly specified


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [8]:
class Matrix:
    # It initialize with Matrix(n, m) or Matrix([[...], [...]])
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            rows, cols = args
            self._rows = rows
            self._cols = cols
            self._data = [[0 for _ in range(cols)] for _ in range(rows)]

        elif len(args) == 1 and isinstance(args[0], list):
            values = args[0]

            if len(values) == 0:
                self._data = []
                self._rows = 0
                self._cols = 0
                return

            col_len = len(values[0])

            # It  make sure each row has the same number of entries
            for row in values:
                if not isinstance(row, list) or len(row) != col_len:
                    raise ValueError("Matrix improperly specified")

            self._data = [row[:] for row in values]
            self._rows = len(values)
            self._cols = col_len

        else:
            raise TypeError("Invalid arguments for Matrix initialization")

    # It supports both M[i][j], M[i, j], and slicing
    def __getitem__(self, index):
        if isinstance(index, tuple):
            r, c = index

            if isinstance(r, slice) or isinstance(c, slice):
                rows = self._data[r]
                new_data = [row[c] for row in rows]
                return Matrix(new_data)

            return self._data[r][c]

        return self._data[index]

    def set_values(self, other):
        if isinstance(other, Matrix):
            new_data = other._data
            new_rows = other._rows
            new_cols = other._cols

        elif isinstance(other, list):
            new_rows = len(other)
            new_cols = len(other[0]) if new_rows > 0 else 0
            new_data = other

        else:
            raise TypeError("Can only assign from Matrix or list of lists")

        if new_rows != self._rows or new_cols != self._cols:
            raise ValueError("Dimensions do not match for assignment")

        self._data = [row[:] for row in new_data]

    def shape(self):
        return (self._rows, self._cols)

    def transpose(self):
        new_data = []
        for j in range(self._cols):
            new_row = []
            for i in range(self._rows):
                new_row.append(self._data[i][j])
            new_data.append(new_row)
        return Matrix(new_data)

    def row(self, n):
        return Matrix([self._data[n][:]])

    def column(self, n):
        new_data = []
        for i in range(self._rows):
            new_data.append([self._data[i][n]])
        return Matrix(new_data)

    def to_list(self):
        return [row[:] for row in self._data]

    def block(self, n_0, n_1, m_0, m_1):
        new_data = []
        for i in range(m_0, m_1):
            new_data.append(self._data[i][n_0:n_1])
        return Matrix(new_data)

    def __str__(self):
        return "\n".join(str(row) for row in self._data)

In [6]:
#Testing
M = Matrix([[1, 2, 3],
            [4, 5, 6],
            [7, 8, 9]])

print("--- Original matrix ---")
print(M)

print("\n--- shape() ---")
print(M.shape())

print("\n--- transpose() ---")
print(M.transpose())

print("\n--- row(1) ---")
print(M.row(1))

print("\n--- column(2) ---")
print(M.column(2))

print("\n--- to_list() ---")
print(M.to_list())

print("\n--- block(0, 2, 1, 3) ---")
print(M.block(0, 2, 1, 3))

print("\n--- slicing with M[0:2, 1:3] ---")
print(M[0:2, 1:3])

--- Original matrix ---
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]

--- shape() ---
(3, 3)

--- transpose() ---
[1, 4, 7]
[2, 5, 8]
[3, 6, 9]

--- row(1) ---
[4, 5, 6]

--- column(2) ---
[3]
[6]
[9]

--- to_list() ---
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]

--- block(0, 2, 1, 3) ---
[4, 5]
[7, 8]

--- slicing with M[0:2, 1:3] ---
[2, 3]
[5, 6]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [10]:
class Matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            rows, cols = args
            self._rows = rows
            self._cols = cols
            self._data = [[0 for _ in range(cols)] for _ in range(rows)]

        elif len(args) == 1 and isinstance(args[0], list):
            values = args[0]

            if len(values) == 0:
                self._data = []
                self._rows = 0
                self._cols = 0
                return

            col_len = len(values[0])
            for row in values:
                if not isinstance(row, list) or len(row) != col_len:
                    raise ValueError("Matrix improperly specified")

            self._data = [row[:] for row in values]
            self._rows = len(values)
            self._cols = col_len

        else:
            raise TypeError("Invalid arguments for Matrix initialization")

    def __str__(self):
        return "\n".join(str(row) for row in self._data)


def constant(n, m, c):
    rows = []
    for _ in range(n):
        rows.append([float(c)] * m)
    return Matrix(rows)


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    rows = []
    for i in range(n):
        row = []
        for j in range(n):
            if i == j:
                row.append(1.0)
            else:
                row.append(0.0)
        rows.append(row)
    return Matrix(rows)

# converting to float so matrix stores consistent values
# using [value] * m to quickly fill each row
# i == j condition is what creates the diagonal in identity matrix
# storing rows first then passing into Matrix constructor

print("--- constant(2, 3, 5) ---")
print(constant(2, 3, 5))

print("\n--- zeros(2, 2) ---")
print(zeros(2, 2))

print("\n--- ones(3, 2) ---")
print(ones(3, 2))

print("\n--- eye(4) ---")
print(eye(4))

--- constant(2, 3, 5) ---
[5.0, 5.0, 5.0]
[5.0, 5.0, 5.0]

--- zeros(2, 2) ---
[0.0, 0.0]
[0.0, 0.0]

--- ones(3, 2) ---
[1.0, 1.0]
[1.0, 1.0]
[1.0, 1.0]

--- eye(4) ---
[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [17]:
class Matrix:
    def __init__(self, *args):
        # making matrix from size n x m
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            rows, cols = args
            self._data = [[0.0 for _ in range(cols)] for _ in range(rows)]
            self._rows = rows
            self._cols = cols

        # making matrix from given list of lists
        elif len(args) == 1 and isinstance(args[0], list):
            values = args[0]
            if not values:
                self._data, self._rows, self._cols = [], 0, 0
                return

            col_len = len(values[0])
            for row in values:
                if not isinstance(row, list) or len(row) != col_len:
                    raise ValueError("Matrix improperly specified")

            self._data = [row[:] for row in values]
            self._rows = len(values)
            self._cols = col_len

        else:
            raise TypeError("Invalid arguments for Matrix initialization")

    def shape(self):
        return (self._rows, self._cols)

    def scalarmul(self, c):
        result = [[value * c for value in row] for row in self._data]
        return Matrix(result)

    def add(self, other):
        if self.shape() != other.shape():
            raise ValueError("Dimensions must match for addition")

        result = [[self._data[i][j] + other._data[i][j] for j in range(self._cols)]
                  for i in range(self._rows)]
        return Matrix(result)

    def sub(self, other):
        if self.shape() != other.shape():
            raise ValueError("Dimensions must match for subtraction")

        result = [[self._data[i][j] - other._data[i][j] for j in range(self._cols)]
                  for i in range(self._rows)]
        return Matrix(result)

    def mat_mult(self, other):
        if self._cols != other._rows:
            raise ValueError("Inner dimensions must match for matrix multiplication")

        result = []
        for i in range(self._rows):
            row_result = []
            for j in range(other._cols):
                total = sum(self._data[i][k] * other._data[k][j] for k in range(self._cols))
                row_result.append(total)
            result.append(row_result)

        return Matrix(result)

    def element_mult(self, other):
        if self.shape() != other.shape():
            raise ValueError("Dimensions must match for element-wise multiplication")

        result = [[self._data[i][j] * other._data[i][j] for j in range(self._cols)]
                  for i in range(self._rows)]
        return Matrix(result)

    def equals(self, other):
        if self.shape() != other.shape():
            return False

        for i in range(self._rows):
            for j in range(self._cols):
                if self._data[i][j] != other._data[i][j]:
                    return False
        return True

    def __str__(self):
        return "\n".join(str(row) for row in self._data)

    # this sum part inside mat_mult is doing the actual row by column multiplication

# for add, sub, and element_mult, shapes need to match or math would not make sense

# equals checks every entry one by one, so if even one value is different it returns False

# quick error test for wrong dimensions
    



In [18]:
#Testing
m1 = Matrix([[1, 2], [3, 4]])
m2 = Matrix([[5, 6], [7, 8]])

print("Matrix m1:")
print(m1)
print("\nMatrix m2:")
print(m2)

print("\n1. Scalar multiplication:")
print(m1.scalarmul(10))

print("\n2. Matrix addition:")
print(m1.add(m2))

print("\n3. Matrix subtraction:")
print(m1.sub(m2))

print("\n4. Matrix multiplication:")
print(m1.mat_mult(m2))

print("\n5. Element-wise multiplication:")
print(m1.element_mult(m2))

print("\n6. Equality check m1 == m2:")
print(m1.equals(m2))

m3 = Matrix([[1, 2], [3, 4]])
print("\n7. Equality check m1 == m3:")
print(m1.equals(m3))


Matrix m1:
[1, 2]
[3, 4]

Matrix m2:
[5, 6]
[7, 8]

1. Scalar multiplication:
[10, 20]
[30, 40]

2. Matrix addition:
[6, 8]
[10, 12]

3. Matrix subtraction:
[-4, -4]
[-4, -4]

4. Matrix multiplication:
[19, 22]
[43, 50]

5. Element-wise multiplication:
[5, 12]
[21, 32]

6. Equality check m1 == m2:
False

7. Equality check m1 == m3:
True


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [21]:
class Matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int):
            r, c = args
            self._rows = r
            self._cols = c
            self._data = [[0.0 for _ in range(c)] for _ in range(r)]

        elif len(args) == 1 and isinstance(args[0], list):
            values = args[0]
            width = len(values[0]) if values else 0

            for row in values:
                if not isinstance(row, list) or len(row) != width:
                    raise ValueError("Matrix improperly specified")

            self._rows = len(values)
            self._cols = width
            self._data = [row[:] for row in values]

        else:
            raise TypeError("Invalid input")

    def shape(self):
        return (self._rows, self._cols)

    def __add__(self, other):
        if self.shape() != other.shape():
            raise ValueError("Dimensions must match for addition")

        out = []
        for i in range(self._rows):
            row = []
            for j in range(self._cols):
                row.append(self._data[i][j] + other._data[i][j])
            out.append(row)
        return Matrix(out)

    def __sub__(self, other):
        if self.shape() != other.shape():
            raise ValueError("Dimensions must match for subtraction")

        out = []
        for i in range(self._rows):
            row = []
            for j in range(self._cols):
                row.append(self._data[i][j] - other._data[i][j])
            out.append(row)
        return Matrix(out)

    def __mul__(self, other):
        # number case
        if type(other) in (int, float):
            out = []
            for row in self._data:
                out.append([x * other for x in row])
            return Matrix(out)

        # matrix case
        if isinstance(other, Matrix):
            if self._cols != other._rows:
                raise ValueError("Inner dimensions must match")

            out = []
            for i in range(self._rows):
                row = []
                for j in range(other._cols):
                    total = 0
                    for k in range(self._cols):
                        total += self._data[i][k] * other._data[k][j]
                    row.append(total)
                out.append(row)
            return Matrix(out)

        return NotImplemented

    def __rmul__(self, other):
        # lets 2 * M work too
        return self * other

    def __eq__(self, other):
        if not isinstance(other, Matrix):
            return False

        if self.shape() != other.shape():
            return False

        # It comparesentry by entry
        for i in range(self._rows):
            for j in range(self._cols):
                if self._data[i][j] != other._data[i][j]:
                    return False
        return True

    def set_values(self, other):
        if self.shape() != other.shape():
            raise ValueError("Dimensions do not match for assignment")
        self._data = [row[:] for row in other._data]

    def __str__(self):
        return "\n".join(str(row) for row in self._data)

In [22]:
#Testing
M = Matrix([[1, 2], [3, 4]])
N = Matrix([[5, 6], [7, 8]])

print("Matrix M:")
print(M)

print("\n1. Check 2 * M:")
print(2 * M)

print("\n2. Check M * 2:")
print(M * 2)

print("\n3. Check M + N:")
print(M + N)

print("\n4. Check M - N:")
print(M - N)

print("\n5. Check M * N:")
print(M * N)

print("\n6. Check M == N:")
print(M == N)

print("\n7. Check M == M:")
print(M == M)

Matrix M:
[1, 2]
[3, 4]

1. Check 2 * M:
[2, 4]
[6, 8]

2. Check M * 2:
[2, 4]
[6, 8]

3. Check M + N:
[6, 8]
[10, 12]

4. Check M - N:
[-4, -4]
[-4, -4]

5. Check M * N:
[19, 22]
[43, 50]

6. Check M == N:
False

7. Check M == M:
True


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [27]:
class Matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int):
            r, c = args
            self._rows = r
            self._cols = c
            self._data = [[0.0 for _ in range(c)] for _ in range(r)]

        elif len(args) == 1 and isinstance(args[0], list):
            vals = args[0]
            col_count = len(vals[0]) if vals else 0

            for item in vals:
                if not isinstance(item, list) or len(item) != col_count:
                    raise ValueError("Matrix improperly specified")

            self._rows = len(vals)
            self._cols = col_count
            self._data = [item[:] for item in vals]

        else:
            raise TypeError("Invalid Matrix input")

    def shape(self):
        return self._rows, self._cols

    def __str__(self):
        return "\n".join(str(r) for r in self._data)

    def __add__(self, other):
        if self.shape() != other.shape():
            raise ValueError("Addition needs same size matrices")

        ans = []
        for i in range(self._rows):
            current = []
            for j in range(self._cols):
                current.append(self._data[i][j] + other._data[i][j])
            ans.append(current)
        return Matrix(ans)

    def __mul__(self, other):
        # regular number multiply
        if isinstance(other, (int, float)):
            ans = []
            for row in self._data:
                ans.append([entry * other for entry in row])
            return Matrix(ans)

        # actual matrix multiply
        if isinstance(other, Matrix):
            if self._cols != other._rows:
                raise ValueError("Matrix sizes do not line up for multiplication")

            ans = []
            for i in range(self._rows):
                current = []
                for j in range(other._cols):
                    value = 0
                    for k in range(self._cols):
                        value += self._data[i][k] * other._data[k][j]
                    current.append(value)
                ans.append(current)
            return Matrix(ans)

        return NotImplemented

    def __rmul__(self, other):
        return self * other

    def __eq__(self, other):
        if not isinstance(other, Matrix):
            return False

        if self.shape() != other.shape():
            return False

        for i in range(self._rows):
            for j in range(self._cols):
                if self._data[i][j] != other._data[i][j]:
                    return False
        return True


def eye(n):
    mat = []
    for i in range(n):
        line = []
        for j in range(n):
            line.append(1.0 if i == j else 0.0)
        mat.append(line)
    return Matrix(mat)


#matrix multiplication done by taking one row from the first matrix
# and one column from the second matrix, then adding those products

# identity matrix works like 1 for matrices, so multiplying by it should not change A

# AB and BA are checked separately here to show matrix multiplication is not commutative


In [26]:
#Testing
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
C = Matrix([[2, 1], [0, 3]])
I = eye(2)

print("A")
print(A)
print("\nB")
print(B)
print("\nC")
print(C)
print("\nI")
print(I)

print("\n" + "-" * 35)
print("1) Check if (AB)C = A(BC)")
side1 = (A * B) * C
side2 = A * (B * C)
print("First side:")
print(side1)
print("Second side:")
print(side2)
print("Match:", side1 == side2)

print("\n" + "-" * 35)
print("2) Check if A(B + C) = AB + AC")
side3 = A * (B + C)
side4 = (A * B) + (A * C)
print("First side:")
print(side3)
print("Second side:")
print(side4)
print("Match:", side3 == side4)

print("\n" + "-" * 35)
print("3) Check if AB = BA")
side5 = A * B
side6 = B * A
print("AB:")
print(side5)
print("BA:")
print(side6)
print("Match:", side5 == side6)

print("\n" + "-" * 35)
print("4) Check if AI = A")
side7 = A * I
print("AI:")
print(side7)
print("A:")
print(A)
print("Match:", side7 == A)

A
[1, 2]
[3, 4]

B
[5, 6]
[7, 8]

C
[2, 1]
[0, 3]

I
[1.0, 0.0]
[0.0, 1.0]

-----------------------------------
1) Check if (AB)C = A(BC)
First side:
[38, 85]
[86, 193]
Second side:
[38, 85]
[86, 193]
Match: True

-----------------------------------
2) Check if A(B + C) = AB + AC
First side:
[21, 29]
[49, 65]
Second side:
[21, 29]
[49, 65]
Match: True

-----------------------------------
3) Check if AB = BA
AB:
[19, 22]
[43, 50]
BA:
[23, 34]
[31, 46]
Match: False

-----------------------------------
4) Check if AI = A
AI:
[1.0, 2.0]
[3.0, 4.0]
A:
[1, 2]
[3, 4]
Match: True
